# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [4]:
# set up environment
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Missing OPENAI_API_KEY in .env")

openai_client = OpenAI()
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

SYSTEM_PROMPT = """You are a patient technical tutor.
Explain clearly and accurately for a programmer learning Python.
Break down what the code does and why. Use short sections or bullet points."""

In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": question.strip()},
]

In [7]:
# Get gpt-4o-mini to answer, with streaming
stream = openai_client.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True,
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

Sure! Let's break down the code you've provided step by step:

### 1. Context
- **`yield from`**: This statement is used in Python generators. It allows the generator to yield all values from an iterable (like a list or a set) in a more concise way than using a loop.
- **`{...}`**: This is a set comprehension, which creates a set of unique values based on the provided iterable.

### 2. Set Comprehension

- **`{book.get("author") for book in books if book.get("author")}`**:
  - **`for book in books`**: This part is iterating over a list or collection called `books`. Each item in this collection is expected to be a dictionary (or an object that supports the `get` method).
  - **`book.get("author")`**: This method attempts to fetch the value associated with the key `"author"` from the `book` dictionary. If `"author"` does not exist in that dictionary, `get` will return `None`.
  - **`if book.get("author")`**: This filter ensures that only those books with a non-null author (i.e., not `None` or an empty string) are included in the set.

### 3. Resulting Set
- The set comprehension will result in a set of unique author values for all books that have a valid `"author"` key. Since it's a set, any duplicate authors will only be represented once.

### 4. Complete Expression
- **`yield from {book.get("author") for book in books if book.get("author")}`**:
  - The generator will yield each of these unique author values one at a time when invoked.
  - This is particularly useful in scenarios where you want to create an iterable of author names without duplicates and return them to the caller one at a time.

### Example
If `books` is defined as follows:
```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3"},  # No author
    {"title": "Book 4", "author": "Author A"},  # Duplicate author
]
```

The code will yield:
- `"Author A"`
- `"Author B"`

### Summary
- The code effectively extracts unique authors from a list of books and provides them generatively. 
- The use of `yield from` makes it efficient for scenarios where you may want to process a large number of authors without loading them all into memory at once.

In [9]:
# Get Llama 3.2 to answer
response = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
)

display(Markdown(response.choices[0].message.content))

**What is this code doing?**

This code is part of a data processing pipeline, likely using Python's generators and iterable data structures. It generates a sequence of author names from a list of books.

**Breakdown:**

* `{book.get("author") for book in books if book.get("author")}` is an **iterable expression**, which creates a sequence from another iterable (in this case, `books`).
	+ The `for book in books` part iterates over the elements in `books`.
	+ The `if book.get("author")` condition filters out any books that don't have an "author" key.
* `yield from ...` is a **generator expression**, which allows us to lazily generate and iterate over the sequence of author names.

**How it contributes to lazy data processing:**

1. When you use `yield from ...`, Python only allocates memory for one element at a time, rather than all elements at once.
2. The iterable expression is evaluated on-the-fly, allowing us to process parts of the data as we need them.

**Why this is useful:**

* Reduces memory usage by not loading large datasets into memory all at once.
* Provides flexible filtering and data processing capabilities using Python's functional programming features.

Here's a simple example:
```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": None},
    {"title": "Book 3", "author": "Author C"}
]

authors = yield from {book.get("author") for book in books if book.get("author")}
print(authors)  # Output: ["Author A", "Author C"]
```
In this example, the `books` list is not fully loaded into memory; instead, we process each book as it's iterated over. The `'yield from ...'` expression still enables us to generate the sequence of author names without loading all books at once.